In [1]:
import torch
from PIL import Image
from diffusers import AutoencoderKL, UNet2DConditionModel, PNDMScheduler
from transformers import CLIPTextModel, CLIPTokenizer



Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [2]:
model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32


In [3]:
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(
    model_id, subfolder="text_encoder", torch_dtype=dtype
).to(device)

vae = AutoencoderKL.from_pretrained(
    model_id, subfolder="vae", torch_dtype=dtype
).to(device)

unet = UNet2DConditionModel.from_pretrained(
    model_id, subfolder="unet", torch_dtype=dtype
).to(device)

scheduler = PNDMScheduler.from_pretrained(model_id, subfolder="scheduler")

vae.eval()
unet.eval()
text_encoder.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

...

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

CLIPTextModel(
  (embeddings): CLIPTextEmbeddings(
    (token_embedding): Embedding(49408, 768)
    (position_embedding): Embedding(77, 768)
  )
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-11): 12 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        )
        (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (final_layer_norm): LayerNo

In [4]:
def encode_prompt(prompt, negative_prompt="", max_length=77):
    text_input = tokenizer(
        [prompt],
        padding="max_length",
        max_length=max_length,
        truncation=True,
        return_tensors="pt",
    )

    uncond_input = tokenizer(
        [negative_prompt],
        padding="max_length",
        max_length=max_length,
        return_tensors="pt",
    )

    with torch.no_grad():
        text_embeds = text_encoder(text_input.input_ids.to(device))[0]
        uncond_embeds = text_encoder(uncond_input.input_ids.to(device))[0]

    return torch.cat([uncond_embeds, text_embeds])

In [5]:
@torch.no_grad()
def generate(
    prompt,
    guidance_scale=7.5,
    height=512,
    width=512,
    num_inference_steps=30,
    seed=42,
    negative_prompt="",
    init_latents=None,
):
    prompt_embeds = encode_prompt(prompt, negative_prompt)

    generator = torch.Generator(device=device).manual_seed(seed)

    if init_latents is None:
        latents = torch.randn(
            (1, unet.config.in_channels, height // 8, width // 8),
            generator=generator,
            device=device,
            dtype=dtype,
        )
        latents = latents * scheduler.init_noise_sigma
    else:
        latents = init_latents.clone().to(device=device, dtype=dtype)

    scheduler.set_timesteps(num_inference_steps, device=device)

    for t in scheduler.timesteps:
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        noise_pred = unet(
            latent_model_input,
            t,
            encoder_hidden_states=prompt_embeds,
        ).sample

        noise_uncond, noise_text = noise_pred.chunk(2)
        noise_pred = noise_uncond + guidance_scale * (noise_text - noise_uncond)

        latents = scheduler.step(noise_pred, t, latents).prev_sample

    latents = latents / vae.config.scaling_factor

    image = vae.decode(latents).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.cpu().permute(0, 2, 3, 1).float().numpy()[0]
    image = (image * 255).round().astype("uint8")

    return Image.fromarray(image)

In [6]:
prompt = "a cinematic photo of a small cabin in a snowy forest, sunset, highly detailed"

for scale in [1.0, 3.0, 7.5, 12.0]:
    img = generate(prompt, guidance_scale=scale, seed=123)
    img.save(f"custom_gs_{scale}.png")

In [7]:
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False,
).to(device)

pipe.scheduler = PNDMScheduler.from_pretrained(model_id, subfolder="scheduler")

prompt = "a cinematic photo of a small cabin in a snowy forest, sunset, highly detailed"
seed = 123
guidance_scale = 7.5
steps = 30

custom_img = generate(
    prompt,
    guidance_scale=guidance_scale,
    num_inference_steps=steps,
    seed=seed,
)

generator = torch.Generator(device=device).manual_seed(seed)

official_img = pipe(
    prompt,
    guidance_scale=guidance_scale,
    num_inference_steps=steps,
    generator=generator,
    height=512,
    width=512,
).images[0]

custom_img.save("custom_pipeline.png")
official_img.save("diffusers_pipeline.png")

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]